In [6]:
!pip install pyspark

In [4]:
from google.colab import files,output

In [3]:
uploaded=files.upload()


Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv


In [8]:
import pandas as pd
import ipywidgets as widgets
import IPython.display as display

In [9]:
output.enable_custom_widget_manager()

In [11]:
from IPython.display import display

file_name=next(iter(uploaded))
df=pd.read_csv(file_name)

print("DataSet connected successfully")
print("Total no of rows: ", len(df))

display(df.head())

DataSet connected successfully
Total no of rows:  12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [12]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [13]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [15]:
filter_df=df.filter(col("quantity") > 3)
print("transformation created")

selected_df=df.select("product","city","price")
print("transformation 2 created")

selected_df.show()

transformation created
transformation 2 created
+-------+---------+-----+
|product|     city|price|
+-------+---------+-----+
| Laptop|Hyderabad|55000|
| Mobile|Bangalore|20000|
| Tablet|  Chennai|15000|
| Mobile|Hyderabad|22000|
| Laptop|   Mumbai|60000|
| Tablet|Bangalore|12000|
| Laptop|  Chennai|58000|
| Mobile|   Mumbai|18000|
| Tablet|Hyderabad|14000|
| Laptop|Bangalore|52000|
| Tablet|   Mumbai|13000|
| Mobile|  Chennai|21000|
+-------+---------+-----+



In [16]:
step1=df.filter(col("quantity") >=3)

step2=df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

step3=step2.groupBy("city").agg(
    sum("revenue").alias("total_revenue")
)

step4=step3.orderBy(desc("total_revenue"))

print("All transformation denied")
step4.show()

All transformation denied
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|       328000|
|   Mumbai|       295000|
|  Chennai|       287000|
|Hyderabad|       268000|
+---------+-------------+



In [17]:
result=df.filter(col("price") >20000) \
        .select("city","product","price") \
        .orderBy(desc("price"))
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=99]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [18]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [20]:
high_value_sales=df.filter(col("price") > 20000)
high_value_sales.show()

print("total records :", high_value_sales.count())

records=high_value_sales.collect()
print(records)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

total records : 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]


In [21]:
from google.colab import files
from pyspark.sql import SparkSession


In [22]:
uploaded=files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


In [23]:
spark = SparkSession.builder.appName("Transformation Example").getOrCreate()

In [24]:
df = spark.read.csv("lazy_execution_sales_25000.csv")

In [29]:
# Read the CSV with headers and inferred types so columns are correctly named and typed
df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)

# Now we can filter using the actual column name 'quantity' which is already typed as an integer
df2 = df.filter(col("quantity") > 5)


In [32]:
df2.select(
    "order_id",
    "product_id" ,
    "quantity",
    "unit_price"
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000020|      P010|       7|      3999|
| O000021|      P006|       7|     55999|
| O000022|      P003|       7|      1899|
| O000023|      P010|       9|      3999|
| O000024|      P009|       7|      1999|
| O000028|      P014|       7|     29999|
| O000033|      P002|       6|      2999|
| O000034|      P012|       6|      6999|
+--------+----------+--------+----

In [33]:
df3=df.filter(
    col("unit_price") > 20000
).count()


In [34]:
df4=df.withColumn(
    "gross_revenue",
    col("quantity") * col("unit_price")
).explain()


== Physical Plan ==
*(1) Project [order_id#291, order_date#292, store_id#293, product_id#294, quantity#295, unit_price#296, discount_pct#297, payment_method#298, order_status#299, sales_channel#300, (quantity#295 * unit_price#296) AS gross_revenue#354]
+- FileScan csv [order_id#291,order_date#292,store_id#293,product_id#294,quantity#295,unit_price#296,discount_pct#297,payment_method#298,order_status#299,sales_channel#300] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...




In [35]:
df4=df.filter(
    col("order_status") == "Completed"
).sort(desc("unit_price")).show(10)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000098|2026-04-15|     S05|      P007|       1|     79999|           5|   Net Banking|   Completed|       Online|
| O000311|2026-03-05|     S06|      P007|       3|     79999|           5|          Card|   Completed|     In Store|
| O000107|2026-05-24|     S05|      P007|       9|     79999|          20|          Cash|   Completed|       Online|
| O000015|2026-04-20|     S08|      P007|       7|     79999|          15|   Net Banking|   Completed|     In Store|
| O000120|2026-05-27|     S03|      P007|       8|     79999|          15|   Net Banking|   Completed|       Online|
| O000009|2026-01-15|     S03|      P007|       5|     79999|   

In [38]:
from pyspark.sql.functions import col

result = df.filter(col("order_status") == "Completed") \
      .withColumn("gross_amount", col("quantity") * col("unit_price")) \
      .select(
          "order_id",
          "quantity",
          "unit_price",
          "gross_amount"
      ) \
      .orderBy(col("gross_amount").desc())

# Single action
result.show()

+--------+--------+----------+------------+
|order_id|quantity|unit_price|gross_amount|
+--------+--------+----------+------------+
| O002603|       9|     79999|      719991|
| O006170|       9|     79999|      719991|
| O002693|       9|     79999|      719991|
| O000107|       9|     79999|      719991|
| O002841|       9|     79999|      719991|
| O000367|       9|     79999|      719991|
| O002940|       9|     79999|      719991|
| O000527|       9|     79999|      719991|
| O003108|       9|     79999|      719991|
| O000690|       9|     79999|      719991|
| O003542|       9|     79999|      719991|
| O001201|       9|     79999|      719991|
| O003884|       9|     79999|      719991|
| O001300|       9|     79999|      719991|
| O003897|       9|     79999|      719991|
| O001796|       9|     79999|      719991|
| O003903|       9|     79999|      719991|
| O002192|       9|     79999|      719991|
| O001170|       9|     79999|      719991|
| O000264|       9|     79999|  

In [39]:
sales_df=spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)
products_df=spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)
stores_df=spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)


In [40]:
sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).select(
    "product_name",
    "category",
    "quantity",
    "unit_price"
).show(20, truncate=False)

+-------------------+-----------+--------+----------+
|product_name       |category   |quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|3       |2999      |
|Smartphone 5G      |Mobiles    |6       |29999     |
|Android Tablet     |Tablets    |6       |22999     |
|Android Tablet     |Tablets    |7       |22999     |
|Laptop 16          |Computers  |2       |79999     |
|Webcam HD          |Accessories|8       |2499      |
|Bluetooth Speaker  |Audio      |8       |3999      |
|Mechanical Keyboard|Accessories|9       |2999      |
|Laptop 16          |Computers  |5       |79999     |
|Laptop 14          |Computers  |5       |55999     |
|Laptop 16          |Computers  |1       |79999     |
|Bluetooth Speaker  |Audio      |9       |3999      |
|External HDD 2TB   |Storage    |6       |5999      |
|Webcam HD          |Accessories|8       |2499      |
|Laptop 16          |Computers  |7       |79999     |
|Laptop 14          |Compute

In [42]:
sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
).groupBy("city").agg(
    sum("quantity").alias("total_quantity")
).show(20, truncate=False)

+---------+--------------+
|city     |total_quantity|
+---------+--------------+
|Chennai  |15727         |
|Mumbai   |15489         |
|Kolkata  |15619         |
|Pune     |15407         |
|Delhi    |15101         |
|Bengaluru|15718         |
|Hyderabad|15784         |
|Jaipur   |16019         |
+---------+--------------+



In [44]:
from pyspark.sql.functions import col, sum

sales_df.join(
    products_df, "product_id"
).join(
    stores_df, "store_id"
).filter(
    col("order_status") == "Completed"
).withColumn(
    "revenue", col("quantity") * col("unit_price")
).groupBy(
    "category", "region"
).agg(
    sum("revenue").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
).show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|  Computers| South|    371954527|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|  Computers|  East|    122990192|
|    Mobiles| South|    120674524|
|    Mobiles|  West|     80372388|
|    Mobiles| North|     75482543|
|    Tablets| South|     64006217|
|   Displays| South|     45761308|
|    Mobiles|  East|     41178126|
|    Tablets| North|     38270336|
|    Tablets|  West|     37488370|
|    Storage| South|     36976319|
|      Audio| South|     34629665|
|   Displays| North|     32468090|
|   Displays|  West|     28286336|
|Accessories| South|     28139018|
|      Audio|  West|     25317111|
|      Audio| North|     24327211|
|    Storage| North|     24150285|
+-----------+------+-------------+
only showing top 20 rows


In [45]:
from pyspark.sql.functions import col, sum

sales_df.filter(
    col("order_status") == "Completed"
).join(
    products_df,
    on="product_id",
    how="inner"
).withColumn(
    "discounted_revenue",
    (col("quantity") * col("unit_price")) *
    (1 - col("discount_pct") / 100)
).groupBy(
    "product_name"
).agg(
    sum("discounted_revenue").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
).show(5, truncate=False)

+---------------+--------------------+
|product_name   |total_revenue       |
+---------------+--------------------+
|Laptop 16      |5.2272546585000014E8|
|Laptop 14      |3.830051604999999E8 |
|Smartphone 5G  |2.060406317500002E8 |
|Android Tablet |1.5040656029999995E8|
|27-inch Monitor|1.1406074015000013E8|
+---------------+--------------------+
only showing top 5 rows


In [48]:
from pyspark.sql.functions import col

# 1. Join with products_df to get product categories, and Filter for 'Completed' orders
df1 = sales_df.join(products_df, on="product_id", how="inner").filter(col("order_status") == "Completed")
print("After join() and filter()")

# 2. Select columns including 'category' instead of non-existent 'product_category'
df2 = df1.select(
    "order_id",
    "category",
    "quantity",
    "unit_price"
)
print("After select()")

# 3. withColumn
df3 = df2.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
print("After withColumn()")

# 4. groupBy
df4 = df3.groupBy("category").sum("gross_amount")
print("After groupBy()")

# 5. orderBy
df5 = df4.orderBy(col("sum(gross_amount)").desc())
print("After orderBy()")

# Action
df5.show()

After join() and filter()
After select()
After withColumn()
After groupBy()
After orderBy()
+-----------+-----------------+
|   category|sum(gross_amount)|
+-----------+-----------------+
|  Computers|        956793873|
|    Mobiles|        317707581|
|    Tablets|        158555106|
|   Displays|        120131933|
|    Storage|         96940081|
|      Audio|         96317122|
|Accessories|         74047394|
+-----------+-----------------+



In [49]:
filtered_df = df.filter(df.order_status == "Completed")

# Show execution plan
filtered_df.explain()

# Action 1
filtered_df.show(5)

# Action 2
filtered_df.count()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#299) AND (order_status#299 = Completed))
+- FileScan csv [order_id#291,order_date#292,store_id#293,product_id#294,quantity#295,unit_price#296,discount_pct#297,payment_method#298,order_status#299,sales_channel#300] Batched: false, DataFilters: [isnotnull(order_status#299), (order_status#299 = Completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,Completed)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------

21775

In [51]:
from pyspark.sql.functions import sum

# Join with products_df to get the 'category' column
result = (
    df.join(products_df, on="product_id", how="inner")
      .groupBy("category")
      .agg(sum("quantity").alias("total_quantity"))
)

data = result.collect()

for row in data:
    print(row)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


In [1]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv

File: lazy_execution_products (1).csv
Records: 15
+----------+-------------------+-----------+----------+----------+
|product_id|       product_name|   category|list_price|cost_price|
+----------+-------------------+-----------+----------+----------+
|      P001|     Wireless Mouse|Accessories|       799|       420|
|      P002|Mechanical Keyboard|Accessories|      2999|      1600|
|      P003|          USB-C Hub|Accessories|      1899|       920|
|      P004|    27-inch Monitor|   Displays|     16999|     11400|
|      P005|          Webcam HD|Accessories|      2499|      1400|
+----------+-------------------+-----------+----------+----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+

In [2]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

completed_df=sales_df.filter(
    F.col("order_status") == "Completed"
)
print("Transfermation completed")
print("total completed orders", completed_df.count())
completed_df.show(10)



Sales file selected: lazy_execution_sales_25000 (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Comple

In [3]:
completed_df.cache()
print("is cached",completed_df.is_cached)

completed_df.count()
completed_df.show(10)

completed_df.groupBy("product_id").count().show()

is cached True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8

In [5]:
high_quantity_df=sales_df.filter(
    F.col("quantity") > 5
)
high_quantity_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

high_quantity_df.count()
high_quantity_df.show(10)

print(high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [6]:
print("Before:",completed_df.is_cached)
completed_df.unpersist(blocking=True)
print("After:",completed_df.is_cached)

high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [7]:
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [8]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")


First Execution: 4.504 seconds
Second Execution: 0.109 seconds


In [9]:
print("total no of records",sales_df.count())

print("total no of partitions:",sales_df.rdd.getNumPartitions())


total no of records 25000
total no of partitions: 1


In [10]:
partitioned_df=sales_df.repartition(8)
print("original no of partitions",sales_df.rdd.getNumPartitions())
print("total no of partitions:",partitioned_df.rdd.getNumPartitions())
partitioned_df.count()

original no of partitions 1
total no of partitions: 8


25000

In [11]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()


+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [12]:
store_partition_df=sales_df.repartition(8, "store_id")
print("original no of partitions",store_partition_df.rdd.getNumPartitions())

store_partition_df.withColumn(
    "partition_id",
    F.spark_partition_id()) \
    .groupBy("partition_id","store_id") \
    .count() \
    .orderBy("partition_id","store_id") \
    .show(30)


original no of partitions 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [13]:
print("Current Partitions:", df.rdd.getNumPartitions())

Current Partitions: 1


In [14]:
df12 = df.repartition(12)

print("Partitions after repartition:", df12.rdd.getNumPartitions())

Partitions after repartition: 12


In [15]:
df6 = df.repartition(6, "product_id")

print("Partitions:", df6.rdd.getNumPartitions())

Partitions: 6


In [16]:
df4 = df12.coalesce(4)

print("Partitions after coalesce:", df4.rdd.getNumPartitions())

Partitions after coalesce: 4
